In [2]:
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters

import warnings
warnings.filterwarnings('ignore')

In [3]:
df_train = pd.read_parquet('../data/raw/X_train.parquet').reset_index()
y_train = pd.read_parquet('../data/raw/y_train.parquet')

# Split the dataset into Pre-break and Post-break segments
df_train_pre = df_train[df_train['period'] == 0][['id', 'time', 'value']]
df_train_post = df_train[df_train['period'] == 1][['id', 'time', 'value']]

In [4]:
# Extract features
features_pre = extract_features(df_train_pre,
                                  column_id='id',
                                  column_sort='time',
                                  default_fc_parameters=EfficientFCParameters())

features_post = extract_features(df_train_post,
                                  column_id='id',
                                  column_sort='time',
                                  default_fc_parameters=EfficientFCParameters())

features_full = extract_features(df_train,
                                column_id='id',
                                column_sort='time',
                                default_fc_parameters=EfficientFCParameters())

Feature Extraction: 100%|██████████| 30/30 [21:19<00:00, 42.65s/it]  


In [5]:
# Cleaning NaNs
print(f'NaN count in the Pre segment:\n{features_pre.isna().sum()[features_pre.isna().sum() > 0]}\n')
print(f'NaN count in the Post segment:\n{features_post.isna().sum()[features_post.isna().sum() > 0]}\n')
print(f'NaN count in the Full series:\n{features_full.isna().sum()[features_full.isna().sum() > 0]}')

features_pre_clean = features_pre.dropna(axis=1, how='all')
features_post_clean = features_post.dropna(axis=1, how='all')
features_clean = features_full.dropna(axis=1, how='all')

NaN count in the Pre segment:
value__query_similarity_count__query_None__threshold_0.0    10001
dtype: int64

NaN count in the Post segment:
value__friedrich_coefficients__coeff_0__m_3__r_30               8
value__friedrich_coefficients__coeff_1__m_3__r_30               8
value__friedrich_coefficients__coeff_2__m_3__r_30               8
value__friedrich_coefficients__coeff_3__m_3__r_30               8
value__max_langevin_fixed_point__m_3__r_30                      8
value__query_similarity_count__query_None__threshold_0.0    10001
dtype: int64

NaN count in the Full series:
period__friedrich_coefficients__coeff_0__m_3__r_30           10001
period__friedrich_coefficients__coeff_1__m_3__r_30           10001
period__friedrich_coefficients__coeff_2__m_3__r_30           10001
period__friedrich_coefficients__coeff_3__m_3__r_30           10001
period__max_langevin_fixed_point__m_3__r_30                  10001
period__query_similarity_count__query_None__threshold_0.0    10001
value__friedrich_

In [6]:
# Add explicit suffixes
df_full_renamed = features_clean.add_suffix('_full')
df_pre_renamed = features_pre_clean.add_suffix('_pre')
df_post_renamed = features_post_clean.add_suffix('_post')


# Calculate the Magic Feature (Global Coefficient of Variation)
if 'value__variance_full' in df_full_renamed.columns and 'value__mean_full' in df_full_renamed.columns:
    std_full = np.sqrt(df_full_renamed['value__variance_full'])
    df_full_renamed['magic_cv_full'] = std_full / (df_full_renamed['value__mean_full'] + 1e-8)
else:
    print("Warning: 'value__variance_full' or 'value__mean_full' not found.")

features_baseline = pd.concat([df_full_renamed, df_pre_renamed, df_post_renamed], axis=1)
features_baseline

,period__variance_larger_than_standard_deviation_full,period__has_duplicate_max_full,period__has_duplicate_min_full,period__has_duplicate_full,period__sum_values_full,period__abs_energy_full,period__mean_abs_change_full,period__mean_change_full,period__mean_second_derivative_central_full,period__median_full,...,value__fourier_entropy__bins_3_post,value__fourier_entropy__bins_5_post,value__fourier_entropy__bins_10_post,value__fourier_entropy__bins_100_post,value__permutation_entropy__dimension_3__tau_1_post,value__permutation_entropy__dimension_4__tau_1_post,value__permutation_entropy__dimension_5__tau_1_post,value__permutation_entropy__dimension_6__tau_1_post,value__permutation_entropy__dimension_7__tau_1_post,value__mean_n_absolute_max__number_of_maxima_7_post
0,0.0,1.0,1.0,1.0,294.0,294.0,0.000608,0.000608,0.0,0.0,...,0.341424,0.795727,1.365678,3.347929,1.783995,3.145679,4.550841,5.384494,5.593754,0.016943
1,0.0,1.0,1.0,1.0,282.0,282.0,0.000396,0.000396,0.0,0.0,...,0.643847,1.140660,1.798221,3.712364,1.790870,3.140268,4.497066,5.354330,5.588368,0.006700
2,0.0,1.0,1.0,1.0,515.0,515.0,0.000351,0.000351,0.0,0.0,...,0.518534,1.042582,1.695340,3.701273,1.789297,3.160725,4.667204,5.801387,6.172530,0.095474
3,0.0,1.0,1.0,1.0,629.0,629.0,0.000345,0.000345,0.0,0.0,...,0.756329,1.277339,1.925920,3.970021,1.789201,3.161664,4.697371,5.932940,6.336889,0.038951
4,0.0,1.0,1.0,1.0,456.0,456.0,0.000473,0.000473,0.0,0.0,...,0.368688,0.603403,1.079170,3.072085,1.769769,3.084793,4.517330,5.658221,6.040310,0.008936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,0.0,1.0,1.0,1.0,554.0,554.0,0.000369,0.000369,0.0,0.0,...,0.843205,1.293286,1.947651,3.918247,1.784900,3.136746,4.615325,5.792523,6.229428,0.026277
9997,0.0,1.0,1.0,1.0,363.0,363.0,0.000467,0.000467,0.0,0.0,...,0.518534,1.004157,1.608351,3.625923,1.787033,3.156134,4.616410,5.578852,5.831138,0.015654
9998,0.0,1.0,1.0,1.0,907.0,907.0,0.000295,0.000295,0.0,0.0,...,0.988099,1.457677,2.134221,4.041587,1.789287,3.167660,4.716495,6.122702,6.708111,0.021337
9999,0.0,1.0,1.0,1.0,881.0,881.0,0.000348,0.000348,0.0,0.0,...,0.965313,1.432970,2.067229,3.949203,1.791265,3.175451,4.733639,6.127119,6.667864,0.004967


In [7]:
features_baseline.to_parquet('../data/processed/features_baseline.parquet')

In [8]:
base_cols = [c.replace('_pre', '') for c in features_baseline.columns if c.endswith('_pre')]

diff_data = {}
ratio_data = {}

for col in base_cols:
    col_pre = f"{col}_pre"
    col_post = f"{col}_post"

    if col_pre in features_baseline.columns and col_post in features_baseline.columns:
        pre = features_baseline[col_pre]
        post = features_baseline[col_post]

        # Calculate Post-Pre Ratios and Diffs
        diff_data[f"{col}_diff"] = post - pre
        ratio_data[f"{col}_ratio"] = post / (pre + 1e-8)

df_diff = pd.DataFrame(diff_data, index=features_baseline.index)
df_ratio = pd.DataFrame(ratio_data, index=features_baseline.index)

# Replace infinities created by zero-division with NaNs
df_ratio = df_ratio.replace([np.inf, -np.inf], np.nan)

features_interactions = pd.concat([df_diff, df_ratio], axis=1)

features_interactions

,value__variance_larger_than_standard_deviation_diff,value__has_duplicate_max_diff,value__has_duplicate_min_diff,value__has_duplicate_diff,value__sum_values_diff,value__abs_energy_diff,value__mean_abs_change_diff,value__mean_change_diff,value__mean_second_derivative_central_diff,value__median_diff,...,value__fourier_entropy__bins_3_ratio,value__fourier_entropy__bins_5_ratio,value__fourier_entropy__bins_10_ratio,value__fourier_entropy__bins_100_ratio,value__permutation_entropy__dimension_3__tau_1_ratio,value__permutation_entropy__dimension_4__tau_1_ratio,value__permutation_entropy__dimension_5__tau_1_ratio,value__permutation_entropy__dimension_6__tau_1_ratio,value__permutation_entropy__dimension_7__tau_1_ratio,value__mean_n_absolute_max__number_of_maxima_7_ratio
0,0.0,0.0,0.0,0.0,-0.017837,-0.052058,-0.000590,6.575549e-05,-3.021947e-05,-0.000217,...,0.402950,0.635880,0.718897,0.858078,0.995827,0.991255,0.958604,0.856379,0.793553,0.755070
1,0.0,0.0,0.0,0.0,-0.313967,-0.013184,-0.000342,6.375096e-06,1.433349e-06,-0.000112,...,0.690426,0.851366,0.922822,0.957867,0.999979,0.990237,0.945541,0.836147,0.752320,0.446551
2,0.0,0.0,0.0,0.0,0.013664,-0.420780,0.005127,-4.499054e-05,-2.309537e-05,0.000648,...,0.690022,0.982218,1.050707,1.018992,0.998828,0.995971,0.979654,0.902845,0.826395,1.179838
3,0.0,0.0,0.0,0.0,-0.660551,-0.105823,0.000792,-8.185923e-06,-2.408034e-06,-0.000264,...,0.805525,1.037589,1.011210,1.020790,0.999847,0.997377,0.987504,0.927322,0.851351,0.881544
4,0.0,0.0,0.0,0.0,0.037261,-0.012898,0.000004,8.533335e-06,3.939975e-06,0.000107,...,0.582620,0.620338,0.689077,0.905631,0.996574,0.989103,0.977390,0.926300,0.860154,0.931517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,0.0,0.0,0.0,0.0,-0.358482,-0.115290,-0.003125,1.851316e-05,-7.867368e-06,0.000581,...,0.919845,1.037384,1.041257,1.008617,0.996489,0.990495,0.975482,0.912817,0.843699,0.519486
9997,0.0,0.0,0.0,0.0,-0.078408,-0.052244,0.000107,-5.670457e-05,8.043478e-06,-0.000105,...,0.632447,0.783942,0.830723,0.939627,0.997813,0.994808,0.971144,0.878595,0.803071,0.826087
9998,0.0,0.0,0.0,0.0,0.058249,-0.083272,-0.000245,7.279720e-06,1.672707e-05,0.000047,...,1.219305,1.139128,1.113977,1.037962,0.998839,0.997783,0.989807,0.954760,0.893680,0.896258
9999,0.0,0.0,0.0,0.0,-0.082905,-0.001739,-0.000271,-7.446796e-07,9.370616e-07,0.000018,...,1.045394,1.216631,1.116447,1.022809,1.000363,1.000549,0.994129,0.958347,0.905325,0.772442


In [9]:
features_interactions.to_parquet('../data/processed/features_interactions.parquet')

In [9]:
df_train = df_train.sort_values(['id', 'time'])

# Global CUMSUM
df_train['cumsum_full'] = df_train.groupby('id')['value'].cumsum()

# Isolated CUMSUM
df_train['cumsum_period'] = df_train.groupby(['id', 'period'])['value'].cumsum()

# Prepare DataFrames
df_cs_full = df_train[['id', 'time', 'cumsum_full']]
df_cs_pre = df_train[df_train['period'] == 0][['id', 'time', 'cumsum_period']]
df_cs_post = df_train[df_train['period'] == 1][['id', 'time', 'cumsum_period']]


# Extract features
feat_cs_pre = extract_features(df_cs_pre, column_id='id', column_sort='time', 
                               column_value='cumsum_period', 
                               default_fc_parameters=EfficientFCParameters())
feat_cs_pre = feat_cs_pre.add_suffix('_cumsum_pre')

feat_cs_post = extract_features(df_cs_post, column_id='id', column_sort='time', 
                                column_value='cumsum_period', 
                                default_fc_parameters=EfficientFCParameters())
feat_cs_post = feat_cs_post.add_suffix('_cumsum_post')

feat_cs_full = extract_features(df_cs_full, column_id='id', column_sort='time', 
                                column_value='cumsum_full', 
                                default_fc_parameters=EfficientFCParameters())
feat_cs_full = feat_cs_full.add_suffix('_cumsum_full')

features_cumsum = pd.concat([feat_cs_pre, feat_cs_post, feat_cs_full], axis=1)
features_cumsum

Feature Extraction: 100%|██████████| 30/30 [11:12<00:00, 22.41s/it]  


,cumsum_period__variance_larger_than_standard_deviation_cumsum_pre,cumsum_period__has_duplicate_max_cumsum_pre,cumsum_period__has_duplicate_min_cumsum_pre,cumsum_period__has_duplicate_cumsum_pre,cumsum_period__sum_values_cumsum_pre,cumsum_period__abs_energy_cumsum_pre,cumsum_period__mean_abs_change_cumsum_pre,cumsum_period__mean_change_cumsum_pre,cumsum_period__mean_second_derivative_central_cumsum_pre,cumsum_period__median_cumsum_pre,...,cumsum_full__fourier_entropy__bins_5_cumsum_full,cumsum_full__fourier_entropy__bins_10_cumsum_full,cumsum_full__fourier_entropy__bins_100_cumsum_full,cumsum_full__permutation_entropy__dimension_3__tau_1_cumsum_full,cumsum_full__permutation_entropy__dimension_4__tau_1_cumsum_full,cumsum_full__permutation_entropy__dimension_5__tau_1_cumsum_full,cumsum_full__permutation_entropy__dimension_6__tau_1_cumsum_full,cumsum_full__permutation_entropy__dimension_7__tau_1_cumsum_full,cumsum_full__query_similarity_count__query_None__threshold_0.0_cumsum_full,cumsum_full__mean_n_absolute_max__number_of_maxima_7_cumsum_full
0,0.0,0.0,0.0,0.0,1.271253,0.103049,0.005592,0.000019,1.105483e-06,0.001056,...,0.539692,0.920663,2.681982,1.755909,3.067438,4.550571,6.019253,6.977872,NaN,0.027519
1,0.0,0.0,0.0,0.0,413.111077,90.887936,0.001775,0.000128,3.507288e-07,0.200143,...,0.090729,0.136002,0.350689,1.725802,2.973130,4.355336,5.733959,6.784726,NaN,0.291612
2,0.0,0.0,0.0,0.0,98.766659,216.382321,0.012565,0.000400,3.595723e-06,-0.014425,...,0.090729,0.136002,0.425614,1.744744,3.016161,4.431826,5.871053,6.979601,NaN,1.868594
3,0.0,0.0,0.0,0.0,667.545585,405.904951,0.006242,0.000394,9.283983e-08,0.181209,...,0.090729,0.136002,0.380783,1.729778,2.972660,4.350202,5.752740,6.852194,NaN,1.168102
4,0.0,0.0,0.0,0.0,-85.773580,8.579513,0.002659,-0.000015,-6.133476e-07,-0.042250,...,0.136002,0.136002,0.470381,1.533610,2.498272,3.530137,4.543425,5.415388,NaN,0.168547
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,0.0,0.0,0.0,0.0,712.108169,268.801653,0.005235,0.000143,1.045423e-07,0.333229,...,0.090729,0.136002,0.472096,1.747344,3.053110,4.519024,6.019870,7.128642,NaN,0.550747
9997,0.0,0.0,0.0,0.0,276.330929,78.036815,0.004808,0.000153,3.262133e-06,0.118212,...,0.136002,0.181214,0.489533,1.738330,2.991218,4.382448,5.757707,6.805033,NaN,0.510084
9998,0.0,0.0,0.0,0.0,-0.309755,0.198239,0.005818,-0.000007,-2.529132e-06,-0.000119,...,0.884647,1.359329,3.203541,1.761416,3.079533,4.581078,6.153990,7.398086,NaN,0.030937
9999,0.0,0.0,0.0,0.0,190.257814,20.654581,0.000808,0.000070,-9.782567e-08,0.098672,...,0.090729,0.136002,0.339942,1.739739,3.009605,4.419511,5.846406,6.951646,NaN,0.199520


In [10]:
features_cumsum.to_parquet('../data/processed/features_cumsum.parquet')